# RoadVision AI: Intelligent Road Condition Classification System for Indian Roads
### Civil Engineering & Artificial Intelligence / Machine Learning Mini Project
**Course Outcomes Addressed:**
- **CO3 (5 Marks):** Problem formulation, dataset preparation (120 Indian records across 10 cities, units, IRC:82 source), model selection (Random Forest), and working classification model.
- **CO4 (5 Marks):** Model evaluation (Confusion Matrix, Precision/Recall/F1, Feature Importance), results interpretation, limitations, and practical Indian Civil Engineering applications.

**Student Group Details:**
- **Student 1:** Rohan Sharma (Roll No: CE-2024-042) — Problem Formulation & Dataset Engineering
- **Student 2:** Ananya Patel (Roll No: CE-2024-058) — Random Forest Model Training & Tuning
- **Student 3:** Aditya Deshmukh (Roll No: CE-2024-071) — Model Evaluation, Web Dashboard & Reporting

## 1. Import Libraries & Load Indian Road Dataset
We load standard scientific Python libraries: `pandas`, `numpy`, `scikit-learn`, and `matplotlib`/`seaborn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load the dataset
df = pd.read_csv('road_condition_dataset.csv')
print(f"Total records loaded: {len(df)}")
print("Sample records across Indian cities:")
df[['record_id', 'city', 'road_segment', 'iri_m_per_km', 'pothole_count', 'condition_category']].head()

## 2. Exploratory Data Analysis & Preprocessing
Inspect the 7 road inspection features and encode categorical variables according to IRC:82 guidelines.

In [ ]:
# Summary statistics
print(df.describe())

# Encode drainage condition: Good=0, Moderate=1, Poor=2
drainage_mapping = {'Good': 0, 'Moderate': 1, 'Poor': 2}
df['drainage_encoded'] = df['drainage_condition'].map(drainage_mapping)

feature_cols = [
    'iri_m_per_km',
    'pothole_count',
    'crack_percentage',
    'rutting_depth_mm',
    'pavement_age_years',
    'drainage_encoded',
    'traffic_volume_vpd'
]

X = df[feature_cols]
y = df['condition_category']

print("Class distribution across Indian roads:")
print(y.value_counts())

## 3. Train-Test Split (80% Training / 20% Testing)
As specified in the syllabus guidelines (CO3), we split the 120 samples into 96 training samples (80%) and 24 testing samples (20%) using stratified sampling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Set: {len(X_train)} samples (80%)")
print(f"Testing Set:  {len(X_test)} samples (20%)")

## 4. Train Random Forest Classifier
Random Forest ensemble of 100 decision trees with max depth 6.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest Model successfully trained on Indian Road dataset!")

## 5. Model Evaluation (CO4: Accuracy, Confusion Matrix, Classification Report)
Evaluating the model on the unseen 20% test partition (24 Indian road samples).

In [ ]:
categories = ['Good', 'Fair', 'Poor', 'Very Poor']
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, labels=categories))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred, labels=categories)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix - Indian Road Condition Classification')
plt.xlabel('Predicted Road Condition')
plt.ylabel('Actual Road Condition')
plt.tight_layout()
plt.show()

## 6. Feature Importance in Indian Pavement Engineering
Analysis of which distress parameters govern Indian road deterioration decisions.

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#f59e0b')
plt.title('Feature Importances in Indian Road Condition Classification')
plt.xlabel('Relative Importance Weight')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Sample Prediction for Field Engineers (e.g. Pune Municipal Corporation)
Predicting a new Indian road segment entering routine inspection.

In [ ]:
# Example input: IRI=4.2 m/km, 3 potholes, 12% cracks, 8.5mm rutting, 4.5 yrs age, Moderate drainage (1), 18000 vpd
sample_road = np.array([[4.2, 3, 12.0, 8.5, 4.5, 1, 18000]])
predicted_condition = rf_model.predict(sample_road)[0]
predicted_proba = rf_model.predict_proba(sample_road)[0]

print(f"Predicted Condition: {predicted_condition}")
for cls, prob in zip(rf_model.classes_, predicted_proba):
    print(f"  {cls}: {prob*100:.1f}%")